# log_parser

In [ ]:
# =================================
#          LOG PARSER
#===================================

import re
import pandas as pd
from datetime import datetime

LOG_PATTERN = re.compile(
    r'(?P<ip>\S+) - - \[(?P<time>.*?)\] "(?P<method>\S+)? (?P<endpoint>\S+)? .*?" (?P<status>\d{3}) (?P<size>\d+) ".*?" "(?P<agent>.*?)"'
)

def parse_log(path):
  rows =[]
  with open(path) as f:
    for line in f:
      match = re.search(LOG_PATTERN, line)
      if match:
        data = match.groupdict()
        data['timestamp'] = datetime.strptime(data['time'], '%d/%b/%Y:%H:%M:%S %z')
        rows.append(data)

  df = pd.DataFrame(rows)
  return df[['timestamp', "ip", "method", "endpoint", "status", "size", "agent"]]

if __name__ == '__main__':
  df = parse_log('access.log')
  print(df.head())
  print(df.info())

  total_lines = len(df)
  print(f"Total lines: {total_lines}")

  top_endpoints = df['endpoint'].value_counts().head(5)
  print(f'top 5 endpoints : {top_endpoints}')

  status = df['status'].value_counts()
  print(f'status : {status}')

                  timestamp               ip method                endpoint  \
0 2025-07-07 00:01:41+00:00   43.129.169.161   HEAD   /Core/Skin/Login.aspx   
1 2025-07-07 00:01:42+00:00   43.129.169.161   HEAD   /Core/Skin/Login.aspx   
2 2025-07-07 00:07:32+00:00  117.198.203.191    GET              /api/user/   
3 2025-07-07 00:07:32+00:00  117.198.203.191    GET         /api/user/info/   
4 2025-07-07 00:07:33+00:00  117.198.203.191    GET  /api/cards/banner-ads/   

  status size                                              agent  
0    301    0  Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...  
1    404    0  Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...  
2    200   52  culturemaxapk/1 CFNetwork/3826.500.131 Darwin/...  
3    200  115  culturemaxapk/1 CFNetwork/3826.500.131 Darwin/...  
4    200   52  culturemaxapk/1 CFNetwork/3826.500.131 Darwin/...  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 194 entries, 0 to 193
Data columns (total 7 columns):
 #   Column     Non

In [ ]:
#=====================
#   EXPAND DATA
#=====================

from datetime import timedelta
import pandas as pd

df = parse_log("access.log")

expanded = []
for i in range(200):  # 194 × 200 ≈ 38,800 rows
    temp = df.copy()
    temp["timestamp"] = temp["timestamp"] + timedelta(minutes=i)
    expanded.append(temp)

big_df = pd.concat(expanded, ignore_index=True)
big_df.to_csv("processed_logs.csv", index=False)

print("Final rows:", len(big_df))

Final rows: 38800


In [ ]:
#==========================
#   PARSING EXPANDED LOGS
#==========================

df = pd.read_csv('/content/processed_logs.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])

print(df.head(5))

total_lines = len(df)
print(f"Total lines: {total_lines}")
print("\n")
top_endpoints = df['endpoint'].value_counts().head(5)
print(f'top 5 endpoints : {top_endpoints}')
print("\n")
status = df['status'].value_counts()
print(f'status : {status}')

                  timestamp               ip method                endpoint  \
0 2025-07-07 00:01:41+00:00   43.129.169.161   HEAD   /Core/Skin/Login.aspx   
1 2025-07-07 00:01:42+00:00   43.129.169.161   HEAD   /Core/Skin/Login.aspx   
2 2025-07-07 00:07:32+00:00  117.198.203.191    GET              /api/user/   
3 2025-07-07 00:07:32+00:00  117.198.203.191    GET         /api/user/info/   
4 2025-07-07 00:07:33+00:00  117.198.203.191    GET  /api/cards/banner-ads/   

   status  size                                              agent  
0     301     0  Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...  
1     404     0  Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...  
2     200    52  culturemaxapk/1 CFNetwork/3826.500.131 Darwin/...  
3     200   115  culturemaxapk/1 CFNetwork/3826.500.131 Darwin/...  
4     200    52  culturemaxapk/1 CFNetwork/3826.500.131 Darwin/...  
Total lines: 38800


top 5 endpoints : endpoint
/api/cards/play-and-sponsor/    8400
/api/user/info/           

# FEATURE ENGINEERING

In [ ]:
import pandas as pd
import numpy as np

def engineer_features(df, window="1min"):
  df = df.sort_values(by="timestamp")
  df["timestamp"] = pd.to_datetime(df["timestamp"])
  df['time_diff'] = df.groupby('ip')['timestamp'].diff().dt.total_seconds()

  endpoint_freq = df['endpoint'].value_counts(normalize=True)
  df['is_rare_endpoint'] = df['endpoint'].map(endpoint_freq) < 0.01

  agg = df.groupby(
      ['ip',
      pd.Grouper(key='timestamp', freq=window)]
  )

  features = agg.agg(
      request_count = ('endpoint', 'count'),
      unique_endpoints = ('endpoint', 'nunique'),
      mean_time_diff = ('time_diff', 'mean'),
      std_time_diff = ('time_diff', 'std'),
      error_rate = ('status', lambda x: np.mean(x>= 400)),
      rare_endpoint_ratio = ('is_rare_endpoint', 'mean'),
      avg_response_size = ('size', 'mean'),
      response_size_std = ('size', 'std')
  ).fillna(0)

  return features.reset_index()


if __name__ == "__main__":
  df = pd.read_csv("/content/processed_logs.csv", parse_dates = ['timestamp'])
  features = engineer_features(df)
  print(features.head())

                ip                 timestamp  request_count  unique_endpoints  \
0  117.198.203.191 2025-07-07 00:07:00+00:00              6                 6   
1  117.198.203.191 2025-07-07 00:08:00+00:00             11                 7   
2  117.198.203.191 2025-07-07 00:09:00+00:00             11                 7   
3  117.198.203.191 2025-07-07 00:10:00+00:00             11                 7   
4  117.198.203.191 2025-07-07 00:11:00+00:00             11                 7   

   mean_time_diff  std_time_diff  error_rate  rare_endpoint_ratio  \
0        0.600000       0.547723         0.0             0.000000   
1        5.454545      12.044614         0.0             0.090909   
2        5.454545      12.044614         0.0             0.090909   
3        5.454545      12.044614         0.0             0.090909   
4        5.454545      12.044614         0.0             0.090909   

   avg_response_size  response_size_std  
0         324.333333         325.201271  
1         242.

In [ ]:
#===================================
#   CHECKING FEATURE ENGINEERING
#===================================

import pandas as pd

# Load processed logs
df = pd.read_csv("processed_logs.csv", parse_dates=["timestamp"])

# Generate features
features = engineer_features(df)

# 1️⃣ Total number of feature rows
print("Total feature rows:", len(features))

# 2️⃣ Find suspicious behavior
suspicious = features[
    (features["request_count"] > features["request_count"].quantile(0.95)) &
    (features["mean_time_diff"] < features["mean_time_diff"].quantile(0.05)) &
    (features["rare_endpoint_ratio"] > 0)
].sort_values("request_count", ascending=False)

print("\n--- Suspicious example ---")
print(suspicious.head(1))

# 3️⃣ Find normal behavior
normal = features[
    (features["request_count"] < features["request_count"].quantile(0.50)) &
    (features["error_rate"] < 0.1) &
    (features["rare_endpoint_ratio"] == 0)
]

print("\n--- Normal example ---")
print(normal.head(1))


Total feature rows: 6542

--- Suspicious example ---
                ip                 timestamp  request_count  unique_endpoints  \
3525  49.36.50.232 2025-07-07 04:52:00+00:00            105                16   

      mean_time_diff  std_time_diff  error_rate  rare_endpoint_ratio  \
3525        0.571429       0.939078    0.038095              0.07619   

      avg_response_size  response_size_std  
3525       20405.857143       94834.429243  

--- Normal example ---
                  ip                 timestamp  request_count  \
1002  196.251.70.164 2025-07-07 02:20:00+00:00              1   

      unique_endpoints  mean_time_diff  std_time_diff  error_rate  \
1002                 1             0.0            0.0         0.0   

      rare_endpoint_ratio  avg_response_size  response_size_std  
1002                  0.0              178.0                0.0  


# RULE BASED BASELINE

In [ ]:
#===========================
#   RULE-BASE
#===========================

def rule_base_score(row):
  score = 0

  if row['request_count'] > 60 :
    score += 2

  if row["unique_endpoints"] > 10:
    score += 2

  if row["mean_time_diff"] < 1:
    score += 2

  if row["error_rate"] > 0.3:
    score += 2

  if row["rare_endpoint_ratio"] > 0.05:
    score += 2

    return int(score)

df = pd.read_csv('/content/processed_logs.csv', parse_dates=['timestamp'])
features = engineer_features(df)

features['rule_score'] = features.apply(rule_base_score, axis=1)
features['rule_flag'] = features['rule_score'] >= 6

print(features['rule_flag'].value_counts())
print(features[features['rule_flag'] == True].head())



rule_flag
False    6341
True      201
Name: count, dtype: int64
                ip                 timestamp  request_count  unique_endpoints  \
3522  49.36.50.232 2025-07-07 04:49:00+00:00             66                12   
3523  49.36.50.232 2025-07-07 04:50:00+00:00            105                16   
3524  49.36.50.232 2025-07-07 04:51:00+00:00            105                16   
3525  49.36.50.232 2025-07-07 04:52:00+00:00            105                16   
3526  49.36.50.232 2025-07-07 04:53:00+00:00            105                16   

      mean_time_diff  std_time_diff  error_rate  rare_endpoint_ratio  \
3522        0.909091       1.652292    0.060606             0.060606   
3523        0.571429       0.939078    0.038095             0.076190   
3524        0.571429       0.939078    0.038095             0.076190   
3525        0.571429       0.939078    0.038095             0.076190   
3526        0.571429       0.939078    0.038095             0.076190   

      avg_respon

In [ ]:
#=================================
#   CHECKING RULE-BASE
#=================================

print(features['rule_flag'].value_counts())
print(f"flagged rows :{features[features['rule_flag'] == True].head(3)}")
print(f"unflagged rows :{features[features['rule_flag'] == False].head(3)}")

rule_flag
False    6341
True      201
Name: count, dtype: int64
flagged rows :                ip                 timestamp  request_count  unique_endpoints  \
3522  49.36.50.232 2025-07-07 04:49:00+00:00             66                12   
3523  49.36.50.232 2025-07-07 04:50:00+00:00            105                16   
3524  49.36.50.232 2025-07-07 04:51:00+00:00            105                16   

      mean_time_diff  std_time_diff  error_rate  rare_endpoint_ratio  \
3522        0.909091       1.652292    0.060606             0.060606   
3523        0.571429       0.939078    0.038095             0.076190   
3524        0.571429       0.939078    0.038095             0.076190   

      avg_response_size  response_size_std  rule_score  rule_flag  
3522       24999.151515      103845.751823         8.0       True  
3523       20405.857143       94834.429243         8.0       True  
3524       20405.857143       94834.429243         8.0       True  
unflagged rows :                ip  

# ISOLATION FOREST (UNSUPERVISED ML)

In [ ]:
from sklearn.ensemble import IsolationForest

df = pd.read_csv('/content/processed_logs.csv', parse_dates=['timestamp'])
features = engineer_features(df)

x = features[
    [
        "request_count",
        "unique_endpoints",
        "mean_time_diff",
        "std_time_diff",
        "error_rate",
        "rare_endpoint_ratio",
        "avg_response_size",
        "response_size_std",
    ]
]

iso = IsolationForest(contamination=0.02, # expect ~2% anomalies
                      n_estimators=200,
                      random_state=42)

features['iso_score'] = iso.fit_predict(x)
features['amomaly_flag'] = features['iso_score'] == -1 # -1 means anomaly

print(features['amomaly_flag'].value_counts())
print(features[features['amomaly_flag']].head(3))


amomaly_flag
False    6537
True        5
Name: count, dtype: int64
                ip                 timestamp  request_count  unique_endpoints  \
3522  49.36.50.232 2025-07-07 04:49:00+00:00             66                12   
3717  49.36.50.232 2025-07-07 08:04:00+00:00             99                13   
3718  49.36.50.232 2025-07-07 08:05:00+00:00             99                13   

      mean_time_diff  std_time_diff  error_rate  rare_endpoint_ratio  \
3522        0.909091       1.652292    0.060606             0.060606   
3717        0.595960       0.957454    0.040404             0.080808   
3718        0.606061       0.977483    0.040404             0.080808   

      avg_response_size  response_size_std  iso_score  amomaly_flag  
3522       24999.151515      103845.751823         -1          True  
3717       21621.515152       97560.563335         -1          True  
3718       21621.515152       97560.563335         -1          True  


In [ ]:
print('=======anomaly count===========')
print(features['amomaly_flag'].value_counts())

features['rule_score'] = features.apply(rule_base_score, axis=1)
features['rule_flag'] = features['rule_score'] >= 6

both_flagged = features[(features['amomaly_flag'] == True) & (features['rule_flag'] == True)]
print('\n=====flagged by RULE-BASE and ML====')
print(both_flagged.head(3))

print('\n=====flagged by ML====')
print(features[features['amomaly_flag'] == True].head(3))

=======anomaly count===========
amomaly_flag
False    6537
True        5
Name: count, dtype: int64

=====flagged by RULE-BASE and ML====
                ip                 timestamp  request_count  unique_endpoints  \
3522  49.36.50.232 2025-07-07 04:49:00+00:00             66                12   
3717  49.36.50.232 2025-07-07 08:04:00+00:00             99                13   
3718  49.36.50.232 2025-07-07 08:05:00+00:00             99                13   

      mean_time_diff  std_time_diff  error_rate  rare_endpoint_ratio  \
3522        0.909091       1.652292    0.060606             0.060606   
3717        0.595960       0.957454    0.040404             0.080808   
3718        0.606061       0.977483    0.040404             0.080808   

      avg_response_size  response_size_std  iso_score  amomaly_flag  \
3522       24999.151515      103845.751823         -1          True   
3717       21621.515152       97560.563335         -1          True   
3718       21621.515152       97560.

# FUSION SCORING ( RULE-BASED  and ML)

In [ ]:
def compute_risk(row):
    risk = 0

    # Rule contribution (max 60%)
    risk += min(row["rule_score"], 10) * 6

    # ML contribution (max 40%)
    if row["amomaly_flag"]:
        risk += 40

    return min(risk, 100)

features['fusion_score'] = features.apply(compute_risk, axis=1)
features['fusion_flag'] = features['fusion_score'] >= 60

print(features['fusion_flag'].value_counts())
print('\n=================================')
print(features[features['fusion_flag']].head(2))
print('\n=================================')
print(features['fusion_score'].describe())
print('\n=================================')
print(features.sort_values('fusion_score', ascending=False).head(5))

fusion_flag
False    6537
True        5
Name: count, dtype: int64

                ip                 timestamp  request_count  unique_endpoints  \
3522  49.36.50.232 2025-07-07 04:49:00+00:00             66                12   
3717  49.36.50.232 2025-07-07 08:04:00+00:00             99                13   

      mean_time_diff  std_time_diff  error_rate  rare_endpoint_ratio  \
3522        0.909091       1.652292    0.060606             0.060606   
3717        0.595960       0.957454    0.040404             0.080808   

      avg_response_size  response_size_std  iso_score  amomaly_flag  \
3522       24999.151515      103845.751823         -1          True   
3717       21621.515152       97560.563335         -1          True   

      rule_score  rule_flag  fusion_score  fusion_flag  
3522         8.0       True          88.0         True  
3717         8.0       True          88.0         True  

count    1402.000000
mean       19.032810
std        13.144573
min        12.000000
25

In [ ]:
suspicious_ip = features[features['fusion_flag'] == True]
print(suspicious_ip.head())

for index, row in suspicious_ip.iterrows():
  print(f"IP: {row['ip']}, Request Count: {row['request_count']}, Fusion Score: {row['fusion_score']}, Rare Endpoint Ratio: {row['rare_endpoint_ratio']}")

                ip                 timestamp  request_count  unique_endpoints  \
3522  49.36.50.232 2025-07-07 04:49:00+00:00             66                12   
3717  49.36.50.232 2025-07-07 08:04:00+00:00             99                13   
3718  49.36.50.232 2025-07-07 08:05:00+00:00             99                13   
3719  49.36.50.232 2025-07-07 08:06:00+00:00             86                13   
3720  49.36.50.232 2025-07-07 08:07:00+00:00             69                12   

      mean_time_diff  std_time_diff  error_rate  rare_endpoint_ratio  \
3522        0.909091       1.652292    0.060606             0.060606   
3717        0.595960       0.957454    0.040404             0.080808   
3718        0.606061       0.977483    0.040404             0.080808   
3719        0.697674       1.117453    0.034884             0.093023   
3720        0.724638       1.679321    0.028986             0.101449   

      avg_response_size  response_size_std  iso_score  amomaly_flag  \
3522     

❓ Why not deep learning?

“Log-based intrusion detection is low-signal and behavior-driven. Tree-based and isolation methods are more interpretable, faster to retrain, and more robust with limited data.”

❓ Why Isolation Forest?

“It works without labels, scales well, and explicitly models anomaly isolation — ideal for security logs.”

❓ Why rules first?

“Rules establish a strong baseline and provide explainability. ML then adds adaptability for unknown patterns.”

❓ What breaks first in production?

“Concept drift — traffic behavior changes. That’s why retraining and monitoring feature distributions are critical.”

🧪 REFINEMENT PASS 4 — VISUAL PROOF (OPTIONAL BUT STRONG)

Add one notebook or script that plots:

risk score distribution

timeline of risk per IP

Not for ML — for trust.

Security teams want to see behavior.

✅ FINAL VERDICT

Dhruv, after refinement this project is:

✔ internship-ready
✔ interview-defensible
✔ system-level (not model-level)
✔ better than 90% of AI student repos